# RLHF Full Validation Evaluation and Example Curation

This notebook is for the **post-training inspection phase**. It does not retrain SFT, reward model, or PPO. It loads the existing checkpoints produced by the repository scripts and runs base-vs-PPO evaluation on the full HelpSteer3 validation preference split.

Outputs are written to:

```text
outputs/rlhf/qwen25_05b_helpsteer3_eval_full/
```

The CSV and JSONL files preserve full prompts and full generations. The Markdown demo is intentionally truncated for readability, so use the JSONL/CSV loading helpers below when curating poster-child examples.

## 0. Colab/repo setup

Run this notebook from the repository root. On Colab, mount Drive first, then `cd` into the cloned/extracted `trpo` repository before running the setup cell.

In [ ]:
# Optional Colab setup. Uncomment only if needed.
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os
import sys

# If this notebook is inside notebooks/, move to repo root.
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

REPO_ROOT = Path.cwd()
print('Repo root:', REPO_ROOT)

# Make the repo importable even before pip install -e .
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# Install once per Colab runtime if needed.
# !pip install -q -r requirements-rlhf.txt
# !pip install -q -e .

## 1. Imports from the RLHF codebase

These are the same code paths used by the CLI scripts under `scripts/`. The main evaluation function is `run_before_after_eval`, which loads the base Qwen model, PPO checkpoint, reference model, reward model, and HelpSteer3 validation prompts.

In [ ]:
import json
import math
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

from trpo_repro.config import load_config, save_config
from trpo_repro.rlhf.evaluate import run_before_after_eval
from trpo_repro.rlhf.metrics import read_jsonl

BASE_EVAL_CONFIG = Path('configs/rlhf/qwen25_05b_helpsteer3_eval.yaml')
FULL_EVAL_CONFIG = Path('configs/rlhf/qwen25_05b_helpsteer3_eval_full.yaml')
FULL_EVAL_DIR = Path('outputs/rlhf/qwen25_05b_helpsteer3_eval_full')

print('Base eval config:', BASE_EVAL_CONFIG.resolve())
print('Full eval config:', FULL_EVAL_CONFIG.resolve())
print('Full eval output dir:', FULL_EVAL_DIR.resolve())

## 2. Full evaluation on the HelpSteer3 validation split

This uses the new config `configs/rlhf/qwen25_05b_helpsteer3_eval_full.yaml`, which inherits the normal eval config but sets `eval.num_prompts: all` and writes to a separate output directory.

This will take longer than the 200-sample evaluation because it generates both base and PPO responses for the full validation split.

In [ ]:
RUN_FULL_EVAL = True  # Set False if the full eval artifacts already exist and you only want analysis/curation.

if RUN_FULL_EVAL:
    out_dir = run_before_after_eval(FULL_EVAL_CONFIG)
    print('Full evaluation saved to:', Path(out_dir).resolve())
else:
    out_dir = FULL_EVAL_DIR
    print('Skipping generation; using existing outputs from:', out_dir.resolve())

## 3. Load full evaluation artifacts

Use `before_after_samples.jsonl` as the source of truth for full text. The CSV also stores full text, but JSONL is safer for nested/newline-heavy model outputs. The Markdown file is intentionally shortened to keep it readable.

In [ ]:
FULL_JSONL = FULL_EVAL_DIR / 'before_after_samples.jsonl'
FULL_CSV = FULL_EVAL_DIR / 'before_after_samples.csv'
FULL_SUMMARY = FULL_EVAL_DIR / 'eval_summary.json'

rows = read_jsonl(FULL_JSONL)
df = pd.DataFrame(rows)

print('Loaded rows:', len(df))
print('Columns:', list(df.columns))

if FULL_SUMMARY.exists():
    summary = json.loads(FULL_SUMMARY.read_text(encoding='utf-8'))
    display(summary)
else:
    summary = {}

df.head(3)

## 4. Aggregate metrics and sanity checks

This section gives the high-level answer: how many examples PPO wins, where it wins by domain/language, how reward deltas are distributed, and whether generations are unexpectedly empty or truncated.

In [ ]:
def add_analysis_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['prompt', 'base_response', 'ppo_response']:
        out[col] = out[col].fillna('').astype(str)
        out[f'{col}_chars'] = out[col].map(len)
        out[f'{col}_words'] = out[col].map(lambda x: len(x.split()))
    for col in ['base_reward', 'ppo_reward', 'reward_delta']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['abs_reward_delta'] = out['reward_delta'].abs()
    out['is_ppo_win'] = out['winner'].eq('ppo')
    out['is_base_win'] = out['winner'].eq('base')
    out['is_near_tie'] = out['reward_delta'].abs() <= 0.25
    return out

adf = add_analysis_columns(df)

print('Examples:', len(adf))
print('\nWinner counts:')
display(adf['winner'].value_counts(dropna=False).to_frame('count'))

print('\nOverall reward stats:')
display(adf[['base_reward', 'ppo_reward', 'reward_delta', 'base_response_chars', 'ppo_response_chars']].describe().T)

print('\nDomain x winner:')
domain_winners = pd.crosstab(adf['domain'], adf['winner'], margins=True)
display(domain_winners)

print('\nLanguage x winner, top 20 languages by count:')
lang_winners = pd.crosstab(adf['language'], adf['winner'])
lang_winners['total'] = lang_winners.sum(axis=1)
display(lang_winners.sort_values('total', ascending=False).head(20))

In [ ]:
# Empty/very short generation sanity checks.
short = adf[(adf['ppo_response_chars'] < 20) | (adf['base_response_chars'] < 20)][
    ['idx', 'domain', 'language', 'winner', 'base_response_chars', 'ppo_response_chars', 'reward_delta']
]
print('Short/empty response rows:', len(short))
display(short.head(30))

In [ ]:
# CJK / non-ASCII / simple pattern checks. These are heuristics, not final safety labels.
CJK_RE = re.compile(r'[㐀-䶿一-鿿豈-﫿]')
BAD_PATTERNS = [
    r'erot', r'adult', r' sex ', r'porn', r'cunt', r'busty', r'voyeur', r'hooker', r'libertin',
    r'blackColor', r'didReceiveMemoryWarning', r'SimpleName', r'HTTPHeader', r'numel',
]

def has_cjk(text):
    return bool(CJK_RE.search(str(text)))

def bad_hits(text):
    lower = ' ' + str(text).lower() + ' '
    return [pat for pat in BAD_PATTERNS if re.search(pat, lower)]

adf['ppo_has_cjk'] = adf['ppo_response'].map(has_cjk)
adf['base_has_cjk'] = adf['base_response'].map(has_cjk)
adf['ppo_bad_hits'] = adf['ppo_response'].map(bad_hits)
adf['base_bad_hits'] = adf['base_response'].map(bad_hits)
adf['ppo_bad_hit_count'] = adf['ppo_bad_hits'].map(len)
adf['base_bad_hit_count'] = adf['base_bad_hits'].map(len)

print('PPO CJK response count:', int(adf['ppo_has_cjk'].sum()))
print('PPO bad-pattern count:', int((adf['ppo_bad_hit_count'] > 0).sum()))

flag_cols = ['idx', 'domain', 'language', 'winner', 'reward_delta', 'ppo_has_cjk', 'ppo_bad_hits', 'prompt']
flagged = adf[(adf['ppo_has_cjk']) | (adf['ppo_bad_hit_count'] > 0)][flag_cols]
display(flagged.head(50))

## 5. Distribution plots

These are quick report/curation plots. They are not saved automatically; save the figures manually from Colab if you want them for the report, or use the code cells as templates.

In [ ]:
plt.figure(figsize=(8, 4))
adf['reward_delta'].hist(bins=50)
plt.axvline(0.0, linestyle='--')
plt.xlabel('PPO reward - base reward')
plt.ylabel('Number of examples')
plt.title('Full validation reward-delta distribution')
plt.show()

plt.figure(figsize=(6, 4))
adf['winner'].value_counts().plot(kind='bar')
plt.xlabel('Winner')
plt.ylabel('Count')
plt.title('Base vs PPO wins')
plt.show()

plt.figure(figsize=(8, 4))
pd.crosstab(adf['domain'], adf['winner']).plot(kind='bar')
plt.xlabel('Domain')
plt.ylabel('Count')
plt.title('Winner counts by domain')
plt.xticks(rotation=30, ha='right')
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(adf['base_reward'], adf['ppo_reward'], alpha=0.45)
mn = min(adf['base_reward'].min(), adf['ppo_reward'].min())
mx = max(adf['base_reward'].max(), adf['ppo_reward'].max())
plt.plot([mn, mx], [mn, mx], linestyle='--')
plt.xlabel('Base reward')
plt.ylabel('PPO reward')
plt.title('Base vs PPO reward scores')
plt.show()

## 6. Full-text example viewer

Use `show_example(idx)` to view the complete prompt, base response, and PPO response. This reads from the JSONL/CSV-loaded DataFrame, not the truncated Markdown demo.

In [ ]:
import html

BOX_STYLE = 'white-space: pre-wrap; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; border: 1px solid #ddd; padding: 12px; border-radius: 6px; background: #fafafa; max-height: 550px; overflow-y: auto;'

def _box(title: str, text: str) -> str:
    safe_title = html.escape(str(title))
    safe_text = html.escape(str(text or ''))
    return f'<h4>{safe_title}</h4><div style="{BOX_STYLE}">{safe_text}</div>'


def show_example(idx: int, source_df: pd.DataFrame = None):
    """Show complete prompt/base/PPO outputs for a row index from the eval file."""
    source_df = adf if source_df is None else source_df
    match = source_df[source_df['idx'].astype(int) == int(idx)]
    if match.empty:
        raise ValueError(f'No row with idx={idx}. Available idx range: {source_df.idx.min()}..{source_df.idx.max()}')
    row = match.iloc[0]
    header = f"""
    <h2>Example {int(row['idx'])}</h2>
    <p><b>Domain:</b> {html.escape(str(row.get('domain', 'unknown')))} &nbsp; 
       <b>Language:</b> {html.escape(str(row.get('language', 'unknown')))} &nbsp; 
       <b>Winner:</b> {html.escape(str(row.get('winner', 'unknown')))}</p>
    <p><b>Base reward:</b> {float(row['base_reward']):.4f} &nbsp; 
       <b>PPO reward:</b> {float(row['ppo_reward']):.4f} &nbsp; 
       <b>Delta:</b> {float(row['reward_delta']):.4f}</p>
    <p><b>Base chars:</b> {int(row['base_response_chars'])} &nbsp; 
       <b>PPO chars:</b> {int(row['ppo_response_chars'])}</p>
    """
    html_out = header + _box('Prompt', row['prompt']) + _box('Base Qwen response', row['base_response']) + _box('PPO-RLHF response', row['ppo_response'])
    display(HTML(html_out))

# Try one example after full eval is loaded.
show_example(int(adf.iloc[0]['idx']))

In [ ]:
# Optional widget browser. If widgets do not render in your Colab session, use show_example(idx) manually.
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    idx_slider = widgets.IntSlider(
        value=int(adf['idx'].min()),
        min=int(adf['idx'].min()),
        max=int(adf['idx'].max()),
        step=1,
        description='idx',
        continuous_update=False,
    )
    out = widgets.Output()

    def _on_change(change):
        with out:
            clear_output(wait=True)
            show_example(change['new'])

    idx_slider.observe(_on_change, names='value')
    display(idx_slider, out)
    with out:
        show_example(idx_slider.value)
except Exception as exc:
    print('Widget browser unavailable:', exc)

## 7. Curation helpers

Use these helpers to find candidate poster-child examples. The reward model is a guide, not a final judge. Manually read candidates before selecting examples for the portfolio/report.

In [ ]:
def candidate_table(kind='ppo_wins', domain=None, language=None, n=20, min_abs_delta=0.0):
    """Return a sorted table of candidate examples for manual review."""
    view = adf.copy()
    if domain is not None:
        view = view[view['domain'].eq(domain)]
    if language is not None:
        view = view[view['language'].eq(language)]
    if min_abs_delta:
        view = view[view['abs_reward_delta'] >= float(min_abs_delta)]

    if kind == 'ppo_wins':
        view = view[view['winner'].eq('ppo')].sort_values('reward_delta', ascending=False)
    elif kind == 'base_wins':
        view = view[view['winner'].eq('base')].sort_values('reward_delta', ascending=True)
    elif kind == 'near_ties':
        view = view[view['is_near_tie']].sort_values('abs_reward_delta', ascending=True)
    elif kind == 'largest_gaps':
        view = view.sort_values('abs_reward_delta', ascending=False)
    elif kind == 'ppo_wins_shorter':
        view = view[view['winner'].eq('ppo')].copy()
        view['ppo_minus_base_chars'] = view['ppo_response_chars'] - view['base_response_chars']
        view = view.sort_values(['reward_delta', 'ppo_minus_base_chars'], ascending=[False, True])
    else:
        raise ValueError("kind must be one of: ppo_wins, base_wins, near_ties, largest_gaps, ppo_wins_shorter")

    cols = ['idx', 'domain', 'language', 'winner', 'base_reward', 'ppo_reward', 'reward_delta', 'base_response_chars', 'ppo_response_chars', 'prompt']
    out = view[cols].head(n).copy()
    out['prompt'] = out['prompt'].str.replace('\n', ' ', regex=False).str.slice(0, 220)
    return out

print('Top PPO wins:')
display(candidate_table('ppo_wins', n=20))

print('Largest base wins / useful failure cases:')
display(candidate_table('base_wins', n=20))

print('Near ties:')
display(candidate_table('near_ties', n=20))

In [ ]:
# Domain-specific candidate views.
for domain in sorted(adf['domain'].dropna().unique()):
    print(f'\n=== Top PPO wins for domain: {domain} ===')
    display(candidate_table('ppo_wins', domain=domain, n=10))

## 8. Save manually selected examples

After reading examples with `show_example(idx)`, put the selected indices into the lists below. This writes full-text JSONL/CSV/Markdown files that we can use later for the report and portfolio.

In [ ]:
# Fill these after manual review.
SELECTED_CLEAR_WINS = []      # e.g., [12, 44, 105]
SELECTED_MIXED_CASES = []     # near ties or ambiguous examples
SELECTED_FAILURES = []        # base clearly better / PPO weakness examples


def _selected_records(indices, label):
    records = []
    for idx in indices:
        row = adf[adf['idx'].astype(int).eq(int(idx))]
        if row.empty:
            print(f'Warning: idx {idx} not found; skipping')
            continue
        rec = row.iloc[0].to_dict()
        rec['curation_label'] = label
        records.append(rec)
    return records


def save_curated_examples(
    clear_wins=SELECTED_CLEAR_WINS,
    mixed=SELECTED_MIXED_CASES,
    failures=SELECTED_FAILURES,
    out_dir=FULL_EVAL_DIR / 'curated_examples',
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    records = []
    records.extend(_selected_records(clear_wins, 'clear_win'))
    records.extend(_selected_records(mixed, 'mixed'))
    records.extend(_selected_records(failures, 'failure'))
    curated = pd.DataFrame(records)
    if curated.empty:
        print('No selected examples yet. Fill the SELECTED_* lists first.')
        return None

    curated.to_csv(out_dir / 'curated_examples.csv', index=False)
    try:
        curated.to_excel(out_dir / 'curated_examples.xlsx', index=False)
    except Exception:
        pass

    with (out_dir / 'curated_examples.jsonl').open('w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    md_lines = ['# Curated RLHF Before/After Examples\n']
    for rec in records:
        md_lines.append(f"## {rec['curation_label']} — idx {int(rec['idx'])} — {rec.get('domain', 'unknown')} / {rec.get('language', 'unknown')}\n")
        md_lines.append(f"**Winner:** {rec.get('winner')}  ")
        md_lines.append(f"**Base reward:** {float(rec['base_reward']):.4f}  ")
        md_lines.append(f"**PPO reward:** {float(rec['ppo_reward']):.4f}  ")
        md_lines.append(f"**Delta:** {float(rec['reward_delta']):.4f}\n")
        md_lines.append('\n### Prompt\n\n')
        md_lines.append(str(rec.get('prompt', '')).strip() + '\n')
        md_lines.append('\n### Base Qwen response\n\n')
        md_lines.append(str(rec.get('base_response', '')).strip() + '\n')
        md_lines.append('\n### PPO-RLHF response\n\n')
        md_lines.append(str(rec.get('ppo_response', '')).strip() + '\n\n---\n')
    (out_dir / 'curated_examples.md').write_text('\n'.join(md_lines), encoding='utf-8')
    print('Saved curated examples to:', out_dir.resolve())
    return curated

# After filling SELECTED_* lists, run:
# curated_df = save_curated_examples()

## 9. Optional: custom prompt comparison

This is optional and separate from the HelpSteer3 validation evaluation. Use it only after the full eval is complete if you want to test your own prompts. It loads the same model checkpoints, so it will use GPU memory.

In [ ]:
# Placeholder for later: we can add custom-prompt comparison here once the full validation curation is done.
# For now, keep the notebook focused on reproducible validation-set evaluation and manual curation.